# 02 — Fix Data: Two Cleaning Strategies

**Strategy A**: remove advertising + duplicates + short body + emoji  
**Strategy B**: Strategy A + remove 'no title' rows (titled articles only)

In [1]:
import sys, os
sys.path.insert(0, '..')
import warnings
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scripts.quality_utils import (
    fix_duplicates, fix_short_body, fix_emoji_titles,
    fix_advertising, fix_dates, run_full_detection, save_report
)

warnings.filterwarnings('ignore')
os.makedirs('../data/cleaned', exist_ok=True)
os.makedirs('../data/reports', exist_ok=True)
print('OK')


OK


## 1. Load original data

In [2]:
df = pd.read_csv('../data/processed/financial_news_merged.csv', encoding='utf-8-sig')
print(f'Original shape: {df.shape}')
print(f'Labeled rows: {df["positive_market_impact"].notna().sum()}')
df.head(3)


Original shape: (15040, 9)
Labeled rows: 15000


,title,body,date,source,sentiment_score,article_type,sectors,tickers,positive_market_impact
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,rdv,-0.5,finance,['Energy'],['GAZP'],0.0
1,no title,""" Рейтинг акций комьюнити РДВ. #опрос """,2024-09-09,rdv,0.0,advertising,[],[],0.0
2,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",2024-09-09,rdv,-0.5,finance,"['Energy', 'Financial Service']",[],0.0


## 2. Strategy A — Conservative Clean

Steps: remove exact duplicates → remove short body → remove advertising → clean emoji titles

In [3]:
df_a = df.copy()
n0 = len(df_a)

# Step 1: exact duplicates
df_a = fix_duplicates(df_a)
print(f'After dedup:        {len(df_a):>6} rows  (removed {n0 - len(df_a)})')

# Step 2: short body
df_a = fix_short_body(df_a, min_len=20)
print(f'After short body:   {len(df_a):>6} rows  (removed {n0 - len(df_a) - 0})')

# Step 3: remove advertising
before_adv = len(df_a)
df_a = fix_advertising(df_a)
print(f'After advertising:  {len(df_a):>6} rows  (removed {before_adv - len(df_a)})')

# Step 4: clean emoji from titles
df_a = fix_emoji_titles(df_a)

# Step 5: parse dates
df_a = fix_dates(df_a)

print(f'\nStrategy A final:   {len(df_a)} rows')
print(f'Loss: {(n0 - len(df_a)) / n0 * 100:.1f}%')
print(f'Labeled: {df_a["positive_market_impact"].notna().sum()}')


After dedup:         15038 rows  (removed 2)
After short body:    15009 rows  (removed 31)
After advertising:   13799 rows  (removed 1210)

Strategy A final:   13799 rows
Loss: 8.3%
Labeled: 13759


In [4]:
df_a.to_csv('../data/cleaned/strategy_a.csv', index=False, encoding='utf-8-sig')
print('Saved: data/cleaned/strategy_a.csv')
df_a.head(3)


Saved: data/cleaned/strategy_a.csv


,title,body,date,source,sentiment_score,article_type,sectors,tickers,positive_market_impact
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,rdv,-0.5,finance,['Energy'],['GAZP'],0.0
1,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",2024-09-09,rdv,-0.5,finance,"['Energy', 'Financial Service']",[],0.0
2,no title,"""️ Ставка ЦБ вряд ли уйдёт выше 18%. ЦБ переже...",2024-09-09,rdv,0.0,opinions,"['Financial Service', 'Real Estate']",['MOEX'],0.0


## 3. Strategy B — Aggressive Clean

Strategy A + remove 'no title' rows → only articles with real titles

In [5]:
df_b = df.copy()
n0 = len(df_b)

# Same steps as A
df_b = fix_duplicates(df_b)
df_b = fix_short_body(df_b, min_len=20)
df_b = fix_advertising(df_b)
df_b = fix_emoji_titles(df_b)
df_b = fix_dates(df_b)

# Extra: remove no-title rows
before_notitle = len(df_b)
df_b = df_b[df_b['title'] != 'no title'].reset_index(drop=True)
print(f'Removed no-title:   {before_notitle - len(df_b)} rows')

print(f'\nStrategy B final:   {len(df_b)} rows')
print(f'Loss: {(n0 - len(df_b)) / n0 * 100:.1f}%')
print(f'Labeled: {df_b["positive_market_impact"].notna().sum()}')


Removed no-title:   9904 rows

Strategy B final:   3895 rows
Loss: 74.1%
Labeled: 3855


In [6]:
df_b.to_csv('../data/cleaned/strategy_b.csv', index=False, encoding='utf-8-sig')
print('Saved: data/cleaned/strategy_b.csv')
df_b.head(3)


Saved: data/cleaned/strategy_b.csv


,title,body,date,source,sentiment_score,article_type,sectors,tickers,positive_market_impact
0,МТС продлевает дополнительный выкуп у нерезиде...,По итогам предыдущего тендерного предложения О...,2024-09-20,finam,-0.5,finance,['Financial Service'],"['EURRUB', 'USDRUB']",0.0
1,ФАС одобрила передачу активов Russ в объединен...,Ведомство не выявило возможного ограничения ко...,2024-09-20,finam,1.0,finance,['Consumer Defensive'],['FIVE'],1.0
2,ЦБ снизил официальный курс доллара на 21 сентя...,"Курс евро увеличен до 103,3773 рубля",2024-09-20,finam,0.0,finance,"['Financial Service', 'Industrials', 'Energy',...",['no tickers'],0.0


## 4. Quality scores after fixing

In [7]:
# Re-read with string columns for quality_utils
df_a_str = pd.read_csv('../data/cleaned/strategy_a.csv', encoding='utf-8-sig')
df_b_str = pd.read_csv('../data/cleaned/strategy_b.csv', encoding='utf-8-sig')

report_a = run_full_detection(df_a_str, 'positive_market_impact')
report_b = run_full_detection(df_b_str, 'positive_market_impact')

save_report(report_a, '../data/reports/quality_report_a.json')
save_report(report_b, '../data/reports/quality_report_b.json')

print(f'Strategy A Quality Score: {report_a["quality_score"]}/100')
print(f'Strategy B Quality Score: {report_b["quality_score"]}/100')


Report saved: ../data/reports/quality_report_a.json
Report saved: ../data/reports/quality_report_b.json
Strategy A Quality Score: 57/100
Strategy B Quality Score: 57/100


## 5. Comparison chart

In [8]:
import json
with open('../data/reports/quality_report.json') as f:
    report_orig = json.load(f)

orig_rows = len(df)
a_rows = len(df_a_str)
b_rows = len(df_b_str)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Row counts
labels = ['Original', 'Strategy A', 'Strategy B']
values = [orig_rows, a_rows, b_rows]
colors = ['#aaaaaa', '#6699cc', '#70b070']
axes[0].bar(labels, values, color=colors, edgecolor='white')
axes[0].set_title('Row count', fontsize=12)
axes[0].set_ylabel('Rows')
for i, v in enumerate(values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Quality scores
scores = [report_orig['quality_score'], report_a['quality_score'], report_b['quality_score']]
axes[1].bar(labels, scores, color=colors, edgecolor='white')
axes[1].set_title('Quality Score', fontsize=12)
axes[1].set_ylabel('Score /100')
axes[1].set_ylim(0, 110)
for i, v in enumerate(scores):
    axes[1].text(i, v + 1, str(v), ha='center', fontsize=10, fontweight='bold')

# Data loss %
losses = [0, (orig_rows - a_rows) / orig_rows * 100, (orig_rows - b_rows) / orig_rows * 100]
axes[2].bar(labels, losses, color=colors, edgecolor='white')
axes[2].set_title('Data loss %', fontsize=12)
axes[2].set_ylabel('Loss %')
for i, v in enumerate(losses):
    axes[2].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/reports/fig_strategy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved fig_strategy_comparison.png')


Saved fig_strategy_comparison.png


## 6. Class balance after fixing

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df_s, title in [
    (axes[0], df_a_str, 'Strategy A'),
    (axes[1], df_b_str, 'Strategy B')
]:
    labeled = df_s[df_s['positive_market_impact'].notna()]
    vc = labeled['positive_market_impact'].value_counts().sort_index()
    ax.bar(['Neg/Neutral (0)', 'Positive (1)'], vc.values,
           color=['#e07070', '#70b070'], edgecolor='white')
    ax.set_title(f'{title} — class balance', fontsize=12)
    ax.set_ylabel('Count')
    for i, v in enumerate(vc.values):
        ax.text(i, v + 10, f'{v}\n({v/len(labeled)*100:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/reports/fig_balance_after_fix.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Summary

In [10]:
orig_rows = len(df)
print('='*55)
print('FIX SUMMARY')
print('='*55)
print(f'Original:   {orig_rows:>6} rows   score={report_orig["quality_score"]}/100')
print(f'Strategy A: {a_rows:>6} rows   score={report_a["quality_score"]}/100   loss={((orig_rows-a_rows)/orig_rows*100):.1f}%')
print(f'Strategy B: {b_rows:>6} rows   score={report_b["quality_score"]}/100   loss={((orig_rows-b_rows)/orig_rows*100):.1f}%')
print()
print('Strategy A removes: duplicates + short body + advertising + emoji')
print('Strategy B removes: all of A + no-title rows (Telegram posts)')


FIX SUMMARY
Original:    15040 rows   score=43/100
Strategy A:  13799 rows   score=57/100   loss=8.3%
Strategy B:   3895 rows   score=57/100   loss=74.1%

Strategy A removes: duplicates + short body + advertising + emoji
Strategy B removes: all of A + no-title rows (Telegram posts)
